In [1]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [2]:
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
device

'mps'

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.shape

(569, 33)

In [5]:
df.drop(['id', 'Unnamed: 32'], axis=1, inplace=True)

In [6]:
df.head(2)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,M,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,...,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,M,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902


In [7]:
df.iloc[1:3, 1:]

,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.186,0.2750,0.08902
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.243,0.3613,0.08758


In [8]:
df.iloc[1, 0]

'M'

In [9]:
xTrain, xTest, yTrain, yTest = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [10]:
scaler = StandardScaler()
xTrain = scaler.fit_transform(xTrain)
xTest = scaler.transform(xTest)

In [12]:
encoder = LabelEncoder()
yTrain = encoder.fit_transform(yTrain)
yTest = encoder.transform(yTrain)

In [13]:
type(xTrain)

numpy.ndarray

In [14]:
xTrain = torch.from_numpy(xTrain)
xTest = torch.from_numpy(xTest)
yTrain = torch.from_numpy(yTrain)
yTest = torch.from_numpy(yTest)

In [15]:
print(type(xTest))
print(type(yTrain))

<class 'torch.Tensor'>
<class 'torch.Tensor'>


In [16]:
xTrain.dtype

torch.float64

In [17]:
xTrain = xTrain.to(torch.float32)
xTest = xTest.to(torch.float32)

In [18]:
yTrain = yTrain.to(torch.float32)

In [19]:
from torch.utils.data import Dataset, DataLoader

In [20]:
class CustomDataLoader(Dataset):
    def __init__(self, features, label):
        self.features = features
        self.label = label

    def __len__(self):
        return self.features.shape[1]

    def __getitem__(self, index):
        return self.features[index], self.label[index]

In [21]:
data = CustomDataLoader(xTrain, yTrain)
test_data = CustomDataLoader(xTest, yTest)

dataloader = DataLoader(data, batch_size=32, shuffle=True)
testDataLoader = DataLoader(test_data, batch_size=32, shuffle=False)

In [22]:
import torch.nn as nn

In [23]:
class SimpleNeuralNetwork(nn.Module):

    def __init__(self, num_features):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(in_features=num_features, out_features=5),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.Linear(5, 1),
            nn.Sigmoid()
        )

    def forward(self, features):
        predict = self.network(features)
        return predict

In [24]:
model = SimpleNeuralNetwork(xTrain.shape[1]).to(device)

In [29]:
epochs = 10
lr = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=1e-2)
loss_fn = nn.BCELoss()

In [30]:
for epoch in range(epochs):
    
    for feature, label in dataloader:

        feature, label = feature.to(device), label.to(device)

        ## Forward Propagation
        yPred = model(feature)

        ## Calculate Loss
        loss = loss_fn(yPred, label.reshape(-1, 1))

        ## clear gradient
        optimizer.zero_grad()

        ## Backpropagation
        loss.backward()

        ## Update parameter
        optimizer.step()

    print(f"Epochs: {epoch + 1}, Loss: {loss.item()}")

Epochs: 1, Loss: 0.4429110586643219
Epochs: 2, Loss: 0.4192134141921997
Epochs: 3, Loss: 0.3966952860355377
Epochs: 4, Loss: 0.3755325376987457
Epochs: 5, Loss: 0.35577839612960815
Epochs: 6, Loss: 0.3374357521533966
Epochs: 7, Loss: 0.32066839933395386
Epochs: 8, Loss: 0.30539777874946594
Epochs: 9, Loss: 0.29138484597206116
Epochs: 10, Loss: 0.2783449590206146


In [31]:
model.eval()

SimpleNeuralNetwork(
  (network): Sequential(
    (0): Linear(in_features=30, out_features=5, bias=True)
    (1): Dropout(p=0.2, inplace=False)
    (2): ReLU()
    (3): Linear(in_features=5, out_features=1, bias=True)
    (4): Sigmoid()
  )
)

In [32]:
total = 0
correct = 0


with torch.no_grad():
    for batch, label in testDataLoader:
        batch, label = batch.to(device), label.to(device)
        predict = model(batch)
        _, output = torch.max(predict, 1)
        total += batch.shape[0]
        correct += (output == label).sum().item()

print(total/correct)

1.6666666666666667
